### **ACID - PART 4a - Compute background function**

# --- --- ---

### This notebook computes and saves a background function. Part 4b notebook imports the background function and uses it for illumination correction.

### Skeep this notebook if a background function has already been computed and saved.

# --- --- ---

**Author:** Alessandro Ulivi (alessandro.ulivi.89@gmail.com)

**Last update (yyyy/mm/dd):** 2026/03/06


## **---------------------------**

### Import required modules

Run the following cell.

Don't modify the following cell.

In [1]:
# Import required modules
import datetime
import os
from pathlib import Path
import numpy as np
import pandas as pd
import napari
from utils.listdirNHF import listdirNHF
from utils.get_defaults import default_file_name
from utils.fov_axis_utils import get_fov_ch_shape
from utils.save_image import tifffile_save_ometiff
from image_processing.calculate_background_function import import_fov, calculate_background_function, calculate_bg_funct_per_condition, get_polyfit_bg_funct_channel



### Specify the paths to input and output data directories - specify hyperparameters

Run the following cell.

Some parts of the following cell should be modified. The parts NOT to modify are under the line ""--- --- --- DON'T MODIFY THE FOLLOWING LINES --- --- ---"

In [30]:
# indicate the path to the directory storing the metadata file - NOTE: this is expected to be the metadata_df saved
# from part1 notebook
metadata_directory = r"Z:\AlessandroUlivi_Data\projects\ACID\data\proc\proc_metadata"
# metadata_directory = r"C:\Users\aless\OneDrive\Desktop\Ale\lab\CIID_IDIP\projects\ACID\data\proc\proc_metadata"

# indicate the path to the directory storing the fields of view
fov_directory = r"Z:\AlessandroUlivi_Data\projects\ACID\data\proc\fov"
# fov_directory = r"C:\Users\aless\OneDrive\Desktop\Ale\lab\CIID_IDIP\projects\ACID\data\proc\fov"

# # indicate the path to the directory where outputs will be saved
# NOTE: it the directory does not exist, the pipeline will try to create it
output_directory = r"Z:\AlessandroUlivi_Data\projects\ACID\data\proc\background"
# output_directory = r"C:\Users\aless\OneDrive\Desktop\Ale\lab\CIID_IDIP\projects\ACID\proc\background"


# name of the metadata file - NOTE: this is expected to be the metadata_df saved
# from part3 notebook
# the following options are possible:
# 1) write the full name (extension included) of the csv file
# 2) write "default" or any other version with a uppercase letter (e.g. "Default")
# 3) leave an empty string: ""
# 4) write None (NOTE that this is not a string, there are no "", aka it is not "None")
# Options 2,3 and 4 will lead to the same behavior: the most recently saved csv file will be used.
# By default, the timestamp of the last file modification is used. As an alternative it is possible to use the
# date saved in the name (change below parameter metadata_from_file_name to True -
# see utils.get_defaults.default_file_name documentation).
metadata_file_name = "default"


# ""--- --- --- DON'T MODIFY THE FOLLOWING LINES --- --- ---"
# # --- parameters to select the train set ---
# indicate the name of the column indicating whether the row belongs to train or test set
is_train_column="is_train"

# indicate the value signalling that a row (aka a field of view) belongs to the train set
train_val=1

# # --- parameters to select the non flagged fields of view ---
# indicate the name of the column indicating whether the row belongs is or is not flagged
flag_column="flag"

# indicate the value signalling that a row (aka a field of view) is flagged
# NOTE: any value which is not this value will be considered as unflagged - only unflagged fields of view
# will be used in the background function computation
flag_value=1


# # --- parameters used for importing the default metadata dataframe ---
# Indicate the part of the file name to use to select files into metadata_directory to be used for selecting the default
# file
default_metadata_file_target = ".csv" # files with this string in their name will be selected for the default file selection - if None, all files will be selected
default_metadata_file_exclude = None # files with this string in their name will be excluded from the default file selection - if None, no files will be excluded

# Indicate whether to extract the date information from the file name when selecting the default file
# If False, the date will be extracted from the file's last modified timestamp, if True, from the file name.
# If from_file_name is True, the parameters *_default_separator and *_default_date_position
# should be used to indicate, respectively the separator to use for splitting the file name into tokens and
# the position of the token containing the date information. In addition, the date format used to include the date in the file name
# should be indicated using the parameters *_default_date_format.
# For example, if metadata_from_file_name is True, and the file name is "251126_plate_layout.csv",
# "_" is used as separator, 0 as date position, and '%Y%m%d' as date format.
# Ref to utils.get_defaults.default_file_name for more details.
metadata_from_file_name = False

# separator - used to split the file name and extract the information token with the date
# if metadata_from_file_name is True
metadata_default_separator = '_'

# date position - the position of the date information token after splitting the file name using file_name_separator (above)
# used if metadata_from_file_name is True
metadata_default_date_position = 0

# date format - the format used for including the date in the file name to be opened by default
# used if metadata_from_file_name is True
metadata_default_date_format='%Y%m%d'

# reverse - if True, the file with the most recent date will be returned by default - if False, the opposite
metadata_default_reverse = True


# --- parameters for background function calculation ---
# background function computation method - this is the method to use for calculating the
# background function. Possible options are:
# "median" - the background function will be calculated as the median of the pixel values across the fields of view (after applying the flag and train set filters)
# "mean" - the background function will be calculated as the mean of the pixel values across the fields of view (after applying the flag and train set filters)
background_function_method = "median"

# polynomial degree to use for fitting the polynomial surface after calculating the background
# function using the method indicated above (median or mean).
# Recommended to use a degree between 2 and 5 - if the degree is too high, the fitted
# surface may overfit the background function and not generalize well to other fields of view
# NOTE: a background function is calculated for each channel. The same polynomial degree is applied to all channels.
polynomial_order_x = (1,2,2,1,1)
polynomial_order_y = (1,2,2,1,1)

# # If None, all coefficients up to maxiumum kx, ky, ie. up to and including x^kx*y^ky, are considered.
# # If int, coefficients up to a maximum of kx+ky <= order are considered.
# # Ref to image_processing.calculate_background_function.get_polyfit_background_function for more details.
# order = None

# channel axis to be passed to background calculating functions in order to calculate background functions per channel
# this is the axis along which the channels are organized in the field of view arrays.
# For example, if the field of view arrays have shape (channels, height, width), the channel axis is 0.
channel_axis = 0

# field of view column name in the metadata dataframe
fov_column_name = "ome_tif_file_name"

# well column name in the metadata dataframe
well_column_name = "well"

# plate column name in the metadata dataframe
plate_column_name = "experiment"

# within well position column name in the metadata dataframe
# this is used to identify the position of the field of view within the well
# 49 fields of view were acquired per each well, in a 7x7 grid
gridpos_column_name = "scene_name"

# null value to be used when a result can't be computed
null_value = np.nan

# kwargs of np.zeros - np.zeros is used to generate a container array storing all images used to
# calculate the background function - by passing kwargs to it one can define the format of the
# background function (e.g. the dtype) - ref to https://numpy.org/doc/stable/reference/generated/numpy.zeros.html
np_zero_kwargs = None

# sample dataframe - this is used for testing purposes
# if True, only a fraction of the fields of view will be used for calculating the background function, and the results will be saved in a separate directory. This is useful for testing the pipeline on a smaller dataset before running it on the full dataset. The fraction of fields of view to use can be defined using the parameter sample_fraction, and additional kwargs for sampling the dataframe can be passed using sample_kwargs (ref to pandas.DataFrame.sample documentation).
sample_df = False
sample_fraction = 0.5
sample_kwargs = {"random_state": 42}

# axis along which fields of view are stacked before calculating the background function
axis_calc_bg = -1

# indicate whether to print verbose messages during the background function calculation
# the same parameter is passed to all the functions used for calculating the background function,
# so that the messages printed during the different steps of the background function calculation are consistent.
verbose_calc_bg = True


# --- parameters for saving metadata within the background function image ---
# background function image - name of processing date in metadata - this is the name
# of the entry in the metadata dictionary to save within the background function image.
# The entry indicates the date when the background function was calculated
background_img_meta_date_name = "background_funct_date_yymmdd"

# background function image - date format in metadata - this is the format to use for indicating
# the date when the background function was calculated in the metadata saved within the background function image
background_img_meta_date_format = '%y%m%d'

# background function image - project name in metadata - this is the name of the entry
# in the metadata dictionary to save within the background function image. The entry indicates the
# name of the project (e.g. ACID)
background_img_meta_project_name = "project_name"

# background function image - method name in metadata - this is the name of the entry
# in the metadata dictionary to save within the background function image. The entry indicates the
# method used for calculating the background function (e.g. median or mean)
background_img_meta_method_name = "background_funct_method"

# background function image - polynomial degree name in metadata - this is the name of the entry
# in the metadata dictionary to save within the background function image. The entry indicates the
# degree of the polynomial used for fitting the background function (if polynomial fitting is applied)
background_img_meta_poly_degree_name = "background_funct_poly_degree"

# indicate whether to save the background function image with ImageJ compatible
save_imagej_compatible = True



# --- parameters for metadata dataframe updating ---
# column name word separator - this is the separator to use for separating the different parts of
# the column names to be added to the metadata dataframe
column_name_separator = "_"

# channel name separator - this is the separator to use for separating the channel name from the rest of
# the column name to be added to the metadata dataframe.
ch_name_separator = "-"

# background function metadata dataframe - method column name - this is the name of the column to be
# added to the metadata dataframe to indicate the method used for calculating the background function
background_df_method_clm_name = f"background{column_name_separator}funct{column_name_separator}method"

# background function metadata dataframe - polynomial degree column name - this is the name of the column to be added to
# the metadata dataframe to indicate the degree of the polynomial used for fitting the background function
background_df_poly_order_x_clm_name = f"background{column_name_separator}funct{column_name_separator}order{column_name_separator}x"
background_df_poly_order_y_clm_name = f"background{column_name_separator}funct{column_name_separator}order{column_name_separator}y"

# background function metadata dataframe - computation date column name - this is the name of the column to be added
# to the metadata dataframe to indicate the day when the background function was calculated
background_df_date_clm_name = f"background{column_name_separator}funct{column_name_separator}date"

# background function metadata dataframe - computation date format - this is the format to be used for indicating the
# date when the background function was calculated. This is used for saving the date in the metadata dataframe
background_df_meta_date_format = '%y%m%d'


# --- parameters for file saving ---
# separator used for saved file names
save_file_name_separator = '_'

# project
project_name = "ACID"

# include indexes when saving pandas dataframes as csv files
save_csv_index = False # if False, the index will not be saved as a separate column in the csv file

# background function image name - date format - this is the format to use for
# indicating the date when the background function was calculated in the background function image name
background_img_name_date_format = '%Y%m%d'

# background function image name - savingword - this is the word to use in the background
# function image name to indicate that the file is a background function image
background_img_savingword = 'background'
non_polyfit_background_img_savingword = 'bg_nofit'

# background function image name - file suffix - this is the suffix to use for the
# background function image file name
background_img_file_suffix = ".ome.tif"

# background function image - data type - this is the data type to use for saving the background function image
background_img_dtype = np.float32

# background function image - photometric - this is the photometric to use for saving the background function image
background_img_photometric = 'minisblack'

# processing metadata dataframe name - savingword
metadata_savingword = "metadata"

# processing metadata dataframe name - file suffix
metadata_file_suffix = f"part{save_file_name_separator}4a.csv"

# processing metadata dataframe name - date format
metadata_date_format = '%Y%m%d'

# hyperparameters dataframe name - date format
hyperparameters_date_format = '%Y%m%d-%H%M%S'

# hyperparameters dataframe name - savingword
hyperparameters_savingword = "hyperparameters"

# hyperparameters dataframe name - file suffix
hyperparameters_file_suffix = f"part{save_file_name_separator}4a.csv"


# --- parameters for saving secondary information ---
# indicate the name of the directory for saving secondary outputs -
# this directory is used to store the hyperparameters used per each run of the pipeline
secondary_output_directory = "secondary_output"
exist_ok = True # if the secondary output directory already exists, do not raise an error



### Create output directory and secondary output directory if they don't exist

##### Output directory stores the computed background function
##### Secondary output directory is used to store the hyperparameters used per each run of the pipeline

Run the following cell.

Don't modify the following cell.

In [3]:
# create the output_directory if it doesn't exist
if not os.path.exists(output_directory):
    os.makedirs(output_directory, exist_ok=exist_ok)

# create the path to secondary_output directory
secondary_output_path = os.path.join(os.getcwd(), secondary_output_directory)

# create the secondary_output directory if it doesn't exist
if not os.path.exists(secondary_output_path):
    os.makedirs(secondary_output_path, exist_ok=exist_ok)


#### Open the metadata dataframe - this is expected to be the output of part 3

Run the following cell.

Don't modify the following cell.

In [4]:

# check if using the default metadata data frame (the most recently saved)
if metadata_file_name==None or metadata_file_name.lower()=="default" or metadata_file_name=="":
    
    # import target files in the metadata_directory
    metadata_files = listdirNHF(metadata_directory,
                                target=default_metadata_file_target,
                                exclude=default_metadata_file_exclude)
    
    # get the default metadata file
    metadata_file_name = default_file_name(file_list=metadata_files,
                                           from_file_name=metadata_from_file_name,
                                           directory_path=metadata_directory,
                                           separator=metadata_default_separator,
                                           date_position=metadata_default_date_position,
                                           date_format=metadata_default_date_format,
                                           reverse=metadata_default_reverse)
    
    print(f"using {metadata_file_name} as default metadata file")


# open the metadata file
metadata_df_i = pd.read_csv(os.path.join(metadata_directory, metadata_file_name))

# copy metadata_df
metadata_df = metadata_df_i.copy()

# # display the metadata dataframe
# metadata_df



using 20260123_ACID_metadata_part_3.csv as default metadata file


#### Select train data set - NOTE: the metadata_df is updated into a metadata_df which does not contain the test data

Run the following cell.

Don't modify the following cell.

In [5]:
# Select only the train set and update metadata_df
metadata_df = metadata_df[metadata_df[is_train_column] == train_val]

# assert proper selection of train set
assert metadata_df.shape[0] > 0, "No rows in metadata_df belong to the train set."
assert all(metadata_df[is_train_column] == train_val), "Not all rows in metadata_df belong to the train set."

# Display the metadata dataframe
metadata_df


,raw_file_name,scene_name,processing_date_yymmdd,ome_tif_file_name,location,microscope,objective,experiment,condition1,infectious_organism,...,top_percentile_fraction-1,top_percentile_fraction-2,top_percentile_fraction-3,top_percentile_fraction-4,mean_over_std-0,mean_over_std-1,mean_over_std-2,mean_over_std-3,mean_over_std-4,flag
0,H7_DENV2_MOI1_30h_fixed_stained_well1.nd2,A1,251211,H7_DENV2_MOI1_30h_fixed_stained_well1_A07p2_A1...,Center for Integrative Infectious Disease Rese...,Nikon Ti2 - CSU-W1 - spinning disc,Nikon Apochromat Lambda-S 60x/1.40 Oil,A07.2,H7,DENV2,...,0.020022,0.020022,0.022416,0.020002,2.585605,1.195264,1.445345,27.645255,17.653226,0
1,H7_DENV2_MOI1_30h_fixed_stained_well1.nd2,A2,251211,H7_DENV2_MOI1_30h_fixed_stained_well1_A07p2_A2...,Center for Integrative Infectious Disease Rese...,Nikon Ti2 - CSU-W1 - spinning disc,Nikon Apochromat Lambda-S 60x/1.40 Oil,A07.2,H7,DENV2,...,0.020003,0.020005,0.021515,0.020010,2.159470,1.592156,1.751387,25.278685,15.561817,0
2,H7_DENV2_MOI1_30h_fixed_stained_well1.nd2,A5,251211,H7_DENV2_MOI1_30h_fixed_stained_well1_A07p2_A5...,Center for Integrative Infectious Disease Rese...,Nikon Ti2 - CSU-W1 - spinning disc,Nikon Apochromat Lambda-S 60x/1.40 Oil,A07.2,H7,DENV2,...,0.020011,0.020014,0.022187,0.020006,2.179615,1.817711,1.600485,27.096500,15.971792,0
3,H7_DENV2_MOI1_30h_fixed_stained_well1.nd2,A6,251211,H7_DENV2_MOI1_30h_fixed_stained_well1_A07p2_A6...,Center for Integrative Infectious Disease Rese...,Nikon Ti2 - CSU-W1 - spinning disc,Nikon Apochromat Lambda-S 60x/1.40 Oil,A07.2,H7,DENV2,...,0.020000,0.020003,0.021979,0.020007,2.243834,1.904663,1.444320,27.040939,15.403914,0
4,H7_DENV2_MOI1_30h_fixed_stained_well1.nd2,B7,251211,H7_DENV2_MOI1_30h_fixed_stained_well1_A07p2_B7...,Center for Integrative Infectious Disease Rese...,Nikon Ti2 - CSU-W1 - spinning disc,Nikon Apochromat Lambda-S 60x/1.40 Oil,A07.2,H7,DENV2,...,0.020019,0.020011,0.021446,0.020018,2.278106,2.020556,1.607375,27.431643,15.968074,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
818,H7_DENV2_MOI1_40h_fixed_stained_well8.nd2,F3,251211,H7_DENV2_MOI1_40h_fixed_stained_well8_A07p4_F3...,Center for Integrative Infectious Disease Rese...,Nikon Ti2 - CSU-W1 - spinning disc,Nikon Apochromat Lambda-S 60x/1.40 Oil,A07.4,H7,DENV2,...,0.020432,0.020003,0.021571,0.020041,1.960308,19.792109,1.524763,27.921094,13.773708,0
819,H7_DENV2_MOI1_40h_fixed_stained_well8.nd2,G2,251211,H7_DENV2_MOI1_40h_fixed_stained_well8_A07p4_G2...,Center for Integrative Infectious Disease Rese...,Nikon Ti2 - CSU-W1 - spinning disc,Nikon Apochromat Lambda-S 60x/1.40 Oil,A07.4,H7,DENV2,...,0.020045,0.020019,0.021691,0.020070,1.827073,8.612130,1.740165,27.868546,15.414203,0
820,H7_DENV2_MOI1_40h_fixed_stained_well8.nd2,G4,251211,H7_DENV2_MOI1_40h_fixed_stained_well8_A07p4_G4...,Center for Integrative Infectious Disease Rese...,Nikon Ti2 - CSU-W1 - spinning disc,Nikon Apochromat Lambda-S 60x/1.40 Oil,A07.4,H7,DENV2,...,0.020598,0.020002,0.021369,0.020025,1.789514,14.308079,1.693619,27.543277,14.619977,0
821,H7_DENV2_MOI1_40h_fixed_stained_well8.nd2,G6,251211,H7_DENV2_MOI1_40h_fixed_stained_well8_A07p4_G6...,Center for Integrative Infectious Disease Rese...,Nikon Ti2 - CSU-W1 - spinning disc,Nikon Apochromat Lambda-S 60x/1.40 Oil,A07.4,H7,DENV2,...,0.020006,0.020002,0.021918,0.020114,2.301495,5.008123,1.532941,18.554031,17.062070,0


#### Select non flagged fields of view

Run the following cell.

Don't modify the following cell.

In [6]:
# Select only non-flagged rows and update metadata_df
metadata_df_ok = metadata_df[metadata_df[flag_column] != flag_value]

# assert proper selection of train set
assert metadata_df_ok.shape[0] > 0, "No row remains in metadata_df after removing the flagged ones."
assert all(metadata_df_ok[flag_column] != flag_value), "Some row in metadata_df_ok are flagged after selection."

# Display the metadata dataframe
metadata_df_ok

# metadata_df_ok = metadata_df.copy()


,raw_file_name,scene_name,processing_date_yymmdd,ome_tif_file_name,location,microscope,objective,experiment,condition1,infectious_organism,...,top_percentile_fraction-1,top_percentile_fraction-2,top_percentile_fraction-3,top_percentile_fraction-4,mean_over_std-0,mean_over_std-1,mean_over_std-2,mean_over_std-3,mean_over_std-4,flag
0,H7_DENV2_MOI1_30h_fixed_stained_well1.nd2,A1,251211,H7_DENV2_MOI1_30h_fixed_stained_well1_A07p2_A1...,Center for Integrative Infectious Disease Rese...,Nikon Ti2 - CSU-W1 - spinning disc,Nikon Apochromat Lambda-S 60x/1.40 Oil,A07.2,H7,DENV2,...,0.020022,0.020022,0.022416,0.020002,2.585605,1.195264,1.445345,27.645255,17.653226,0
1,H7_DENV2_MOI1_30h_fixed_stained_well1.nd2,A2,251211,H7_DENV2_MOI1_30h_fixed_stained_well1_A07p2_A2...,Center for Integrative Infectious Disease Rese...,Nikon Ti2 - CSU-W1 - spinning disc,Nikon Apochromat Lambda-S 60x/1.40 Oil,A07.2,H7,DENV2,...,0.020003,0.020005,0.021515,0.020010,2.159470,1.592156,1.751387,25.278685,15.561817,0
2,H7_DENV2_MOI1_30h_fixed_stained_well1.nd2,A5,251211,H7_DENV2_MOI1_30h_fixed_stained_well1_A07p2_A5...,Center for Integrative Infectious Disease Rese...,Nikon Ti2 - CSU-W1 - spinning disc,Nikon Apochromat Lambda-S 60x/1.40 Oil,A07.2,H7,DENV2,...,0.020011,0.020014,0.022187,0.020006,2.179615,1.817711,1.600485,27.096500,15.971792,0
3,H7_DENV2_MOI1_30h_fixed_stained_well1.nd2,A6,251211,H7_DENV2_MOI1_30h_fixed_stained_well1_A07p2_A6...,Center for Integrative Infectious Disease Rese...,Nikon Ti2 - CSU-W1 - spinning disc,Nikon Apochromat Lambda-S 60x/1.40 Oil,A07.2,H7,DENV2,...,0.020000,0.020003,0.021979,0.020007,2.243834,1.904663,1.444320,27.040939,15.403914,0
4,H7_DENV2_MOI1_30h_fixed_stained_well1.nd2,B7,251211,H7_DENV2_MOI1_30h_fixed_stained_well1_A07p2_B7...,Center for Integrative Infectious Disease Rese...,Nikon Ti2 - CSU-W1 - spinning disc,Nikon Apochromat Lambda-S 60x/1.40 Oil,A07.2,H7,DENV2,...,0.020019,0.020011,0.021446,0.020018,2.278106,2.020556,1.607375,27.431643,15.968074,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
818,H7_DENV2_MOI1_40h_fixed_stained_well8.nd2,F3,251211,H7_DENV2_MOI1_40h_fixed_stained_well8_A07p4_F3...,Center for Integrative Infectious Disease Rese...,Nikon Ti2 - CSU-W1 - spinning disc,Nikon Apochromat Lambda-S 60x/1.40 Oil,A07.4,H7,DENV2,...,0.020432,0.020003,0.021571,0.020041,1.960308,19.792109,1.524763,27.921094,13.773708,0
819,H7_DENV2_MOI1_40h_fixed_stained_well8.nd2,G2,251211,H7_DENV2_MOI1_40h_fixed_stained_well8_A07p4_G2...,Center for Integrative Infectious Disease Rese...,Nikon Ti2 - CSU-W1 - spinning disc,Nikon Apochromat Lambda-S 60x/1.40 Oil,A07.4,H7,DENV2,...,0.020045,0.020019,0.021691,0.020070,1.827073,8.612130,1.740165,27.868546,15.414203,0
820,H7_DENV2_MOI1_40h_fixed_stained_well8.nd2,G4,251211,H7_DENV2_MOI1_40h_fixed_stained_well8_A07p4_G4...,Center for Integrative Infectious Disease Rese...,Nikon Ti2 - CSU-W1 - spinning disc,Nikon Apochromat Lambda-S 60x/1.40 Oil,A07.4,H7,DENV2,...,0.020598,0.020002,0.021369,0.020025,1.789514,14.308079,1.693619,27.543277,14.619977,0
821,H7_DENV2_MOI1_40h_fixed_stained_well8.nd2,G6,251211,H7_DENV2_MOI1_40h_fixed_stained_well8_A07p4_G6...,Center for Integrative Infectious Disease Rese...,Nikon Ti2 - CSU-W1 - spinning disc,Nikon Apochromat Lambda-S 60x/1.40 Oil,A07.4,H7,DENV2,...,0.020006,0.020002,0.021918,0.020114,2.301495,5.008123,1.532941,18.554031,17.062070,0


#### Get the shape and the number of channels of the fields of view - NOTE: it is assumed that all fields of view in the dataset have the same shape and number of channels

Run the following cell.

Don't modify the following cell.

In [7]:
# get the shape of the individual fields of view, the number of channels and the shape of individual channels
fov_shape, num_channels, shape_of_channels = get_fov_ch_shape(metadata_df_ok,
                                                              fov_directory,
                                                              fov_clm=fov_column_name,
                                                              channel_axis=channel_axis,
                                                              null_value=null_value)


field of view shape: (5, 1024, 1024)
number of channels found: 5
shape of channels: (1024, 1024)


#### Strategy 1 - Calculate a background function by averaging all the fields of view in the dataset

The following cell:
1) import all the fields of view in the dataset
2) stack all the fields of view in a single array.
3) average all the fields of view. Averaging is computed separately per individual channels. Two methods can be used for averaging: median or mean. The method is chosen by the background_function_method hyperparameter.

# --- --- ---

Run the following cell.

Don't modify the following cell.

In [8]:
# import all fields of view in the dataset and stack them into a single array
container_arr = import_fov(df=metadata_df_ok,
                           fov_dir=fov_directory,
                           fov_clm=fov_column_name,
                           fov_shape=fov_shape,
                           np_zero_kwargs=np_zero_kwargs,
                           verbose=verbose_calc_bg,
                           sample_df=sample_df,
                           sample_fraction=sample_fraction,
                           sample_kwargs=sample_kwargs)

# initialize a list for storing the background function calculated for each channel
background_function_collection = []

# unstack the container array along the channel axis to get a list of arrays
# each containing the pixel values of the fields of view for a single channel
# this is used for calculating the background function per channel
container_arr_ch = np.unstack(container_arr, axis=channel_axis)

# iterate over the list of arrays and calculate the background function for each channel
for ch_idx, container_arr_ch_i in enumerate(container_arr_ch):
    print(container_arr_ch_i.shape)
    # calculate the background function for individual channels by mean/median average projection
    ch_i_background_function = calculate_background_function(container_arr=container_arr_ch_i,
                                                           method=background_function_method,
                                                           axis=axis_calc_bg,
                                                           verbose=verbose_calc_bg)
    print(ch_i_background_function.shape)

    # add the calculated background function for the current channel to the list for storing the background function
    background_function_collection.append(ch_i_background_function)

# stack the background function calculated for each channel into a single array
background_function = np.stack(background_function_collection, axis=channel_axis)
print(background_function.shape)


container_arr shape: (5, 1024, 1024, 817)
(1024, 1024, 817)
background_function shape: (1024, 1024)
(1024, 1024)
(1024, 1024, 817)
background_function shape: (1024, 1024)
(1024, 1024)
(1024, 1024, 817)
background_function shape: (1024, 1024)
(1024, 1024)
(1024, 1024, 817)
background_function shape: (1024, 1024)
(1024, 1024)
(1024, 1024, 817)
background_function shape: (1024, 1024)
(1024, 1024)
(5, 1024, 1024)


#### Visualize the background functions per each channel

Run the following cell.

Don't modify the following cell.

In [16]:
# istanziate a napari viewer and add the background function to the viewer
napari_viewer = napari.Viewer()

for ch in range(background_function.shape[channel_axis]):
    napari_viewer.add_image(background_function.take(ch, axis=channel_axis), name=f"background_funct_ch_{ch}")



#### Strategy 2 - Calculate a background function per each well of the dataset

The following cell, iteratively per each well:
    
1) import all the fields of view in the dataset belonging to the well
2) stack the fields of view in a single array.
3) average the fields of view. Averaging is computed separately per individual channels. Two methods can be used for averaging: median or mean. The method is chosen by the background_function_method hyperparameter.

# --- --- ---

Run the following cell.

Don't modify the following cell.

In [10]:

# get the background functions of all the wells in the dataframe and map them into a dictionary
background_functions_per_well = calculate_bg_funct_per_condition(df=metadata_df_ok,
                                                                 condition_clm=well_column_name,
                                                                 fov_dir=fov_directory,
                                                                 fov_clm=fov_column_name,
                                                                 fov_shape=fov_shape,
                                                                 method=background_function_method,
                                                                 stack_axis=axis_calc_bg,
                                                                 verbose=verbose_calc_bg)


unique conditions: ['well1' 'well2' 'well3' 'well4' 'well5' 'well6' 'well7' 'well8']
--- --- ---
--- --- --- well1
container_arr shape: (5, 1024, 1024, 104)
background_function shape: (5, 1024, 1024)
--- --- --- well2
container_arr shape: (5, 1024, 1024, 102)
background_function shape: (5, 1024, 1024)
--- --- --- well3
container_arr shape: (5, 1024, 1024, 104)
background_function shape: (5, 1024, 1024)
--- --- --- well4
container_arr shape: (5, 1024, 1024, 105)
background_function shape: (5, 1024, 1024)
--- --- --- well5
container_arr shape: (5, 1024, 1024, 98)
background_function shape: (5, 1024, 1024)
--- --- --- well6
container_arr shape: (5, 1024, 1024, 104)
background_function shape: (5, 1024, 1024)
--- --- --- well7
container_arr shape: (5, 1024, 1024, 97)
background_function shape: (5, 1024, 1024)
--- --- --- well8
container_arr shape: (5, 1024, 1024, 103)
background_function shape: (5, 1024, 1024)


#### Visualize the background functions per each well and target channels

Run the following cell.

Some parts of the following cell should be modified. The parts NOT to modify are under the line ""--- --- --- DON'T MODIFY THE FOLLOWING LINES --- --- ---"

In [21]:
# indicate the channel to visualize
ch_to_plot = 4

# --- --- --- DON'T MODIFY THE FOLLOWING LINES --- --- ---

# istanziate a napari viewer and add the background function to the viewer
napari_viewer_1 = napari.Viewer()

for wel_l in background_functions_per_well:
    napari_viewer_1.add_image(background_functions_per_well[wel_l].take(ch_to_plot, axis=channel_axis), name=f"background_funct_{wel_l}_{ch_to_plot}")



#### Strategy 3 - Calculate a background function per each field of view grid position

Per each well 49 positions are aquired from a 7x7 grid. The following cells analyse the result of computing a background function per each of the 49 position, by averaging across experiemnts, plates and wells.

Precisely, the following cell, iteratively per each grid position:
    
1) import all the fields of view in the dataset belonging to the grid position
2) stack the fields of view in a single array.
3) average the fields of view. Averaging is computed separately per individual channels. Two methods can be used for averaging: median or mean. The method is chosen by the background_function_method hyperparameter.

# --- --- ---

Run the following cell.

Don't modify the following cell.

In [22]:

# get the background functions of all the wells in the dataframe and map them into a dictionary
background_functions_per_gridpos = calculate_bg_funct_per_condition(df=metadata_df_ok,
                                                                    condition_clm=gridpos_column_name,
                                                                    fov_dir=fov_directory,
                                                                    fov_clm=fov_column_name,
                                                                    fov_shape=fov_shape,
                                                                    method=background_function_method,
                                                                    stack_axis=axis_calc_bg,
                                                                    verbose=verbose_calc_bg)


unique conditions: ['A1' 'A2' 'A5' 'A6' 'B7' 'B6' 'B5' 'B3' 'B1' 'C1' 'C2' 'C3' 'C4' 'C5'
 'C6' 'C7' 'D7' 'D6' 'D4' 'D2' 'E1' 'E2' 'E5' 'E6' 'E7' 'F7' 'F6' 'F5'
 'F4' 'F2' 'F1' 'G1' 'G4' 'G5' 'G7' 'A4' 'B2' 'D3' 'E3' 'E4' 'G2' 'G3'
 'A7' 'B4' 'D5' 'D1' 'F3' 'G6' 'A3']
--- --- ---
--- --- --- A1
container_arr shape: (5, 1024, 1024, 15)
background_function shape: (5, 1024, 1024)
--- --- --- A2
container_arr shape: (5, 1024, 1024, 21)
background_function shape: (5, 1024, 1024)
--- --- --- A5
container_arr shape: (5, 1024, 1024, 17)
background_function shape: (5, 1024, 1024)
--- --- --- A6
container_arr shape: (5, 1024, 1024, 21)
background_function shape: (5, 1024, 1024)
--- --- --- B7
container_arr shape: (5, 1024, 1024, 16)
background_function shape: (5, 1024, 1024)
--- --- --- B6
container_arr shape: (5, 1024, 1024, 19)
background_function shape: (5, 1024, 1024)
--- --- --- B5
container_arr shape: (5, 1024, 1024, 18)
background_function shape: (5, 1024, 1024)
--- --- --- B3
container_a

#### Visualize the background functions per each grid position and target channels

Run the following cell.

Some parts of the following cell should be modified. The parts NOT to modify are under the line ""--- --- --- DON'T MODIFY THE FOLLOWING LINES --- --- ---"

In [27]:
# indicate the channel to visualize
ch_to_plot_1 = 4

# --- --- --- DON'T MODIFY THE FOLLOWING LINES --- --- ---

# istanziate a napari viewer and add the background function to the viewer
napari_viewer_2 = napari.Viewer()

for grid_pos in background_functions_per_gridpos:
    napari_viewer_2.add_image(background_functions_per_gridpos[grid_pos].take(ch_to_plot_1, axis=channel_axis), name=f"background_funct_{grid_pos}_{ch_to_plot_1}")


#### Define the strategy to use for background function calculation

Run the following cell.

MODIFY the following cell.

In [28]:
# indicate which strategy to use for creating the background function by fitting a
# polynomial surface to the averaged background function
background_function_strategy = 1 # pick between 1, 2 or 3


#### Fit polynomial surface to background function. Update and save the metadata_df. NOTE: the update is done on the metadata_df after train data filter and before filtering of the flagged rows.

Run the following cell.

Don't modify the following cell.

In [ ]:

# create the background function by fitting a polynomial surface to the averaged background function and
# according to the strategy indicated above, save the background function as an image
if background_function_strategy == 1:

    # create the background function by fitting a polynomial surface to the averaged background function
    # NOTE: the parameter order is not passed and, therefore, set to None, meaning that all coefficients up to
    # the maximum kx and ky indicated will be considered
    polyfit_background_function = get_polyfit_bg_funct_channel(background_function=background_function,
                                                               channel_axis=channel_axis,
                                                               kx= polynomial_order_x,
                                                               ky= polynomial_order_y,
                                                               verbose= verbose_calc_bg)

    # create the background function image name
    background_img_name = f"{datetime.datetime.now().strftime(background_img_name_date_format)}{save_file_name_separator}{project_name}{save_file_name_separator}{background_img_savingword}{background_img_file_suffix}"
    
    # create the background function image metadata dictionary
    background_img_metadata_dict = {background_img_meta_date_name: datetime.datetime.now().strftime(background_img_meta_date_format),
                                    background_img_meta_project_name: project_name,
                                    background_img_meta_method_name: background_function_method,
                                    background_img_meta_poly_degree_name: f"kx: {polynomial_order_x}, ky: {polynomial_order_y}, order: {None}"} # NOTE: order is hardcoded as None
    
    # save the background function as an image
    tifffile_save_ometiff(os.path.join(output_directory, background_img_name),
                                  data=polyfit_background_function.astype(background_img_dtype),
                                  imagej=save_imagej_compatible,
                                  photometric=background_img_photometric,
                                  metadata=background_img_metadata_dict)
    
    # also save the non-polyfit background function as an image for comparison
    background_img_name_non_polyfit = f"{datetime.datetime.now().strftime(background_img_name_date_format)}{save_file_name_separator}{project_name}{save_file_name_separator}{non_polyfit_background_img_savingword}{background_img_file_suffix}"
    background_img_metadata_dict_non_polyfit = {background_img_meta_date_name: datetime.datetime.now().strftime(background_img_meta_date_format),
                                                background_img_meta_project_name: project_name,
                                                background_img_meta_method_name: background_function_method} # NOTE: order is hardcoded as None

    # save the non-polyfit background function as an image for comparison
    tifffile_save_ometiff(os.path.join(output_directory, background_img_name_non_polyfit),
                          data=background_function.astype(background_img_dtype),
                          imagej=save_imagej_compatible,
                          photometric=background_img_photometric,
                          metadata=background_img_metadata_dict_non_polyfit)

elif background_function_strategy == 2:

    # iterate over the wells
    for wel_l in background_functions_per_well:

        # create the background function by fitting a polynomial surface to the averaged background function of the well
        # NOTE: the parameter order is not passed and, therefore, set to None, meaning that all coefficients up to the maximum
        # kx and ky indicated will be considered
        polyfit_background_function_well = get_polyfit_bg_funct_channel(background_function=background_functions_per_well[wel_l],
                                                                                   channel_axis=channel_axis,
                                                                                   kx= polynomial_order_x,
                                                                                   ky= polynomial_order_y,
                                                                                   verbose= verbose_calc_bg)

        # create the background function image name for the well
        background_img_name_well = f"{datetime.datetime.now().strftime(background_img_name_date_format)}{save_file_name_separator}{project_name}{save_file_name_separator}{background_img_savingword}{save_file_name_separator}{wel_l}{background_img_file_suffix}"
        
        # create the background function image metadata dictionary for the well
        background_img_metadata_dict_well = {background_img_meta_date_name: datetime.datetime.now().strftime(background_img_meta_date_format),
                                            background_img_meta_project_name: project_name,
                                            background_img_meta_method_name: background_function_method,
                                            background_img_meta_poly_degree_name: f"kx: {polynomial_order_x}, ky: {polynomial_order_y}, order: {None}", # NOTE: order is hardcoded as None
                                            "well": wel_l}
        
        # save the background function as an image for the well
        tifffile_save_ometiff(os.path.join(output_directory, background_img_name_well),
                              data=polyfit_background_function_well.astype(background_img_dtype),
                              imagej=save_imagej_compatible,
                              photometric=background_img_photometric,
                              metadata=background_img_metadata_dict_well)

elif background_function_strategy == 3:

    # iterate over the grid positions
    for grid_pos in background_functions_per_gridpos:

        # create the background function by fitting a polynomial surface to the averaged background function of the grid position
        # NOTE: the parameter order is not passed and, therefore, set to None, meaning that all coefficients up
        # to the maximum kx and ky indicated will be considered
        polyfit_background_function_gridpos = get_polyfit_bg_funct_channel(background_function=background_functions_per_gridpos[grid_pos],
                                                                                   channel_axis=channel_axis,
                                                                                   kx= polynomial_order_x,
                                                                                   ky= polynomial_order_y,
                                                                                   verbose= verbose_calc_bg)

        # create the background function image name for the grid position
        background_img_name_gridpos = f"{datetime.datetime.now().strftime(background_img_name_date_format)}{save_file_name_separator}{project_name}{save_file_name_separator}{background_img_savingword}{save_file_name_separator}{grid_pos}{background_img_file_suffix}"

        # create the background function image metadata dictionary for the grid position
        background_img_metadata_dict_gridpos = {background_img_meta_date_name: datetime.datetime.now().strftime(background_img_meta_date_format),
                                               background_img_meta_project_name: project_name,
                                               background_img_meta_method_name: background_function_method,
                                               background_img_meta_poly_degree_name: f"kx: {polynomial_order_x}, ky: {polynomial_order_y}, order: {None}", # NOTE: order is hardcoded as None
                                               "grid_position": grid_pos}

        # save the background function as an image for the grid position
        tifffile_save_ometiff(os.path.join(output_directory, background_img_name_gridpos),
                              data=polyfit_background_function_gridpos.astype(background_img_dtype),
                              imagej=save_imagej_compatible,
                              photometric=background_img_photometric,
                              metadata=background_img_metadata_dict_gridpos)

else:
    raise ValueError("background_function_strategy must be 1, 2 or 3")


# --- --- --- UPDATE THE METADATA DATAFRAME WITH THE BACKGROUND FUNCTION INFORMATION --- --- ---
# add the background function method, polynomial degree and calculation date information to the metadata dataframe
# as new columns
# NOTE: the metadata_df used is the one after selecting the train set but before selecting only the non-flagged
# fields of view
metadata_df[background_df_date_clm_name] = datetime.datetime.now().strftime(background_df_meta_date_format)
metadata_df[background_df_method_clm_name] = background_function_method
for ch_pos, ch_order in enumerate(zip(polynomial_order_x, polynomial_order_y)):
    metadata_df[f"{background_df_poly_order_x_clm_name}{ch_name_separator}{ch_pos}"] = ch_order[0]
    metadata_df[f"{background_df_poly_order_y_clm_name}{ch_name_separator}{ch_pos}"] = ch_order[1]

# save the updated metadata dataframe as a csv file
metadata_df_name = f"{datetime.datetime.now().strftime(metadata_date_format)}{save_file_name_separator}{project_name}{save_file_name_separator}{metadata_savingword}{save_file_name_separator}{metadata_file_suffix}"
metadata_df.to_csv(os.path.join(metadata_directory, metadata_df_name), index=save_csv_index)



Processing channel 0...
polyfit_background_function shape: (1024, 1024)
Processing channel 1...
polyfit_background_function shape: (1024, 1024)
Processing channel 2...
polyfit_background_function shape: (1024, 1024)
Processing channel 3...
polyfit_background_function shape: (1024, 1024)
Processing channel 4...
polyfit_background_function shape: (1024, 1024)


### Save hyperparameters

Run the following cell.

Don't modify the following cell.

In [33]:
# # collect hyperparameters in a dictionary

hyperparameter_dict = {

'metadata_directory': metadata_directory,
'fov_directory': fov_directory,
'output_directory': output_directory,
'metadata_file_name': metadata_file_name,
'is_train_column': is_train_column,
'train_val': train_val,
'flag_column': flag_column,
'flag_value': flag_value,
'default_metadata_file_target': default_metadata_file_target,
'default_metadata_file_exclude': default_metadata_file_exclude,
'metadata_from_file_name': metadata_from_file_name,
'metadata_default_separator': metadata_default_separator,
'metadata_default_date_position': metadata_default_date_position,
'metadata_default_date_format': metadata_default_date_format,
'metadata_default_reverse': metadata_default_reverse,
'background_function_method': background_function_method,
'polynomial_order_x': polynomial_order_x,
'polynomial_order_y': polynomial_order_y,
'channel_axis': channel_axis,
'fov_column_name': fov_column_name,
'well_column_name': well_column_name,
'plate_column_name': plate_column_name,
'gridpos_column_name': gridpos_column_name,
'null_value': null_value,
'np_zero_kwargs': np_zero_kwargs,
'sample_df': sample_df,
'sample_fraction': sample_fraction,
'sample_kwargs': sample_kwargs,
'axis_calc_bg': axis_calc_bg,
'verbose_calc_bg': verbose_calc_bg,
'background_img_meta_date_name': background_img_meta_date_name,
'background_img_meta_date_format': background_img_meta_date_format,
'background_img_meta_project_name': background_img_meta_project_name,
'background_img_meta_method_name': background_img_meta_method_name,
'background_img_meta_poly_degree_name': background_img_meta_poly_degree_name,
'save_imagej_compatible': save_imagej_compatible,
'column_name_separator': column_name_separator,
'ch_name_separator': ch_name_separator,
'background_df_method_clm_name': background_df_method_clm_name,
'background_df_poly_order_x_clm_name': background_df_poly_order_x_clm_name,
'background_df_poly_order_y_clm_name': background_df_poly_order_y_clm_name,
'background_df_date_clm_name': background_df_date_clm_name,
'background_df_meta_date_format': background_df_meta_date_format,
'save_file_name_separator': save_file_name_separator,
'project_name': project_name,
'save_csv_index': save_csv_index,
'background_img_name_date_format': background_img_name_date_format,
'background_img_savingword': background_img_savingword,
'background_img_file_suffix': background_img_file_suffix,
'background_img_dtype': background_img_dtype,
'background_img_photometric': background_img_photometric,
'metadata_savingword': metadata_savingword,
'metadata_file_suffix': metadata_file_suffix,
'metadata_date_format': metadata_date_format,
'hyperparameters_date_format': hyperparameters_date_format,
'hyperparameters_savingword': hyperparameters_savingword,
'hyperparameters_file_suffix': hyperparameters_file_suffix,
'secondary_output_directory': secondary_output_directory,
'exist_ok': exist_ok

}

# transform the hyperparameter_dict in a pandas series
hyperparameter_series = pd.Series(hyperparameter_dict)

# save hyperparamters
hyperparameter_saving_name = f"{datetime.datetime.now().strftime(hyperparameters_date_format)}{save_file_name_separator}{project_name}{save_file_name_separator}{hyperparameters_savingword}{save_file_name_separator}{hyperparameters_file_suffix}"
hyperparameter_series.to_csv(os.path.join(secondary_output_directory,hyperparameter_saving_name), index=save_csv_index)

